# 03 — T1 baselines: collinear vs non-collinear
**Project:** MAG2D-NC | **Phase:** F5 / Step 1 | **Protocol:** v1.1 (frozen)

Task T1: binary classification, `collinear` (79) vs `non_collinear` (85),
164 materials, 61 leakage groups, 144 descriptors.

Contract implemented here (frozen — editing thresholds/metrics after seeing
results is a protocol violation):
- CV: StratifiedGroupKFold(5), 3 repeats x 5 seeds -> 75 outer folds per model
- Nested HP search: RandomizedSearchCV, 50 trials, inner StratifiedGroupKFold(3)
- Models: dummy-majority, dummy-stratified, logistic regression, LightGBM
- Metrics: F1-macro (primary), MCC, balanced accuracy; fold-level bootstrap 95% CI
- Tests: Friedman + Holm-corrected Wilcoxon; McNemar (pooled OOF); Cliff's delta
- Permutation test (v1.1): 1000 label permutations, fixed modal HPs, p=(1+#>=obs)/1001
- Failure criterion P7: best F1-macro must beat stratified dummy with
  Holm-corrected p<0.05 AND Cliff's delta >= 0.33, else T1 is reported as a
  negative finding.

Runtime note: full run is CPU-only, roughly 1-3 h. Set `QUICK_SMOKE=True` first
to verify plumbing in ~2 min; smoke results are never reported.

## CONFIG

In [ ]:
from pathlib import Path
from datetime import datetime
import glob

CONFIG = {
    "PROJECT_ROOT": Path.home() / "MAG2D-NC",
    "SEEDS": [0, 1, 2, 3, 4],
    "N_SPLITS": 5,
    "N_REPEATS": 3,
    "HP_TRIALS": 50,
    "INNER_SPLITS": 3,
    "N_PERMUTATIONS": 1000,
    "N_BOOTSTRAP": 10000,
    "PRIMARY": "f1_macro",
    "RUN_STAMP": datetime.now().strftime("%Y%m%d-%H%M%S"),
}
QUICK_SMOKE = False   # True: tiny budgets for plumbing check only — NEVER report

if QUICK_SMOKE:
    CONFIG.update({"SEEDS":[0], "N_REPEATS":1, "HP_TRIALS":5, "N_PERMUTATIONS":20,
                   "N_BOOTSTRAP":500})
    print("*** SMOKE MODE — results not reportable ***")

feats = sorted(glob.glob(str(CONFIG["PROJECT_ROOT"]/"dataset"/"features_T1_*.parquet")))
assert feats, "No features parquet found — run the feature notebook first."
CONFIG["FEATURES"] = Path(feats[-1])
print("features file:", CONFIG["FEATURES"].name)
print({k: v for k, v in CONFIG.items() if k not in ("PROJECT_ROOT","FEATURES")})

## Dependencies

In [ ]:
import importlib, subprocess, sys
for pkg, mod in [("lightgbm","lightgbm"), ("scikit-learn","sklearn"), ("scipy","scipy")]:
    try:
        importlib.import_module(mod); print(pkg, "OK")
    except ImportError:
        subprocess.run([sys.executable,"-m","pip","install",pkg], check=True)
        print(pkg, "installed")

## Load features + gate

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet(CONFIG["FEATURES"])
assert len(df) == 164 and df["group_id"].nunique() == 61
fcols = [c for c in df.columns if c.startswith(("comp_","soc_","sym_"))]
X = df[fcols].to_numpy(dtype=float)
y = (df["label2"] == "non_collinear").astype(int).to_numpy()
groups = df["group_id"].to_numpy()
print(f"X: {X.shape} | positives (non_collinear): {y.sum()} | negatives: {(1-y).sum()}")
assert (y.sum(), (1-y).sum()) == (85, 79)

## Models + HP spaces
Preprocessing lives INSIDE each pipeline (fit on train folds only — §P1.1).
Equal 50-trial budget per searched model; dummies have nothing to search.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from lightgbm import LGBMClassifier
from scipy.stats import loguniform, randint, uniform

def make_models():
    return {
      "dummy_majority": (DummyClassifier(strategy="most_frequent"), None),
      "dummy_stratified": (DummyClassifier(strategy="stratified", random_state=0), None),
      "logreg": (Pipeline([("vt", VarianceThreshold(0.0)),
                           ("sc", StandardScaler()),
                           ("clf", LogisticRegression(max_iter=5000,
                                                      class_weight="balanced"))]),
                 {"clf__C": loguniform(1e-3, 1e3),
                  "clf__penalty": ["l2"],
                  "clf__solver": ["lbfgs"]}),
      "lgbm": (Pipeline([("vt", VarianceThreshold(0.0)),
                         ("clf", LGBMClassifier(objective="binary", verbosity=-1,
                                                class_weight="balanced",
                                                n_jobs=-1))]),
               {"clf__n_estimators": randint(100, 800),
                "clf__learning_rate": loguniform(0.005, 0.3),
                "clf__num_leaves": randint(4, 64),
                "clf__min_child_samples": randint(3, 30),
                "clf__subsample": uniform(0.6, 0.4),
                "clf__colsample_bytree": uniform(0.5, 0.5),
                "clf__reg_lambda": loguniform(1e-3, 10)}),
    }
print("models:", list(make_models().keys()))

## Outer loop: repeated grouped stratified CV with nested search
75 outer folds per model (5 seeds x 3 repeats x 5 folds). For each fold we log
fold-level metrics and out-of-fold predictions; every row is traceable in
`runs.csv`.

In [ ]:
import time, json as _j
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV
from sklearn.metrics import f1_score, matthews_corrcoef, balanced_accuracy_score
from collections import Counter

OUT = CONFIG["PROJECT_ROOT"] / "output"
OUT.mkdir(exist_ok=True)
runs_path = OUT / "runs.csv"

fold_rows = []          # fold-level metrics
oof = {}                # (model, seed, repeat) -> (y_true_concat, y_pred_concat aligned to df order)
best_params_log = {m: [] for m in ("logreg","lgbm")}

t0 = time.time()
for seed in CONFIG["SEEDS"]:
    for rep in range(CONFIG["N_REPEATS"]):
        rs = 1000*seed + rep
        outer = StratifiedGroupKFold(n_splits=CONFIG["N_SPLITS"], shuffle=True, random_state=rs)
        folds = list(outer.split(X, y, groups))
        for mname, (est, space) in make_models().items():
            y_pred_full = np.full(len(y), -1)
            for k, (tr, te) in enumerate(folds):
                if space is None:
                    model = est
                    model.fit(X[tr], y[tr])
                else:
                    inner = StratifiedGroupKFold(n_splits=CONFIG["INNER_SPLITS"],
                                                 shuffle=True, random_state=rs)
                    search = RandomizedSearchCV(est, space, n_iter=CONFIG["HP_TRIALS"],
                                                scoring="f1_macro",
                                                cv=inner.split(X[tr], y[tr], groups[tr]),
                                                random_state=rs, n_jobs=-1, refit=True)
                    search.fit(X[tr], y[tr])
                    model = search.best_estimator_
                    best_params_log[mname].append(
                        {k2: (round(v,5) if isinstance(v,float) else v)
                         for k2, v in search.best_params_.items()})
                yp = model.predict(X[te])
                y_pred_full[te] = yp
                fold_rows.append({
                    "run_stamp": CONFIG["RUN_STAMP"], "task": "T1", "model": mname,
                    "seed": seed, "repeat": rep, "fold": k,
                    "f1_macro": f1_score(y[te], yp, average="macro"),
                    "mcc": matthews_corrcoef(y[te], yp),
                    "bal_acc": balanced_accuracy_score(y[te], yp),
                    "n_test": len(te),
                })
            assert (y_pred_full >= 0).all()
            oof[(mname, seed, rep)] = y_pred_full
        print(f"seed {seed} rep {rep} done | elapsed {time.time()-t0:.0f}s")

folds_df = pd.DataFrame(fold_rows)
folds_df.to_csv(runs_path, mode="a", header=not runs_path.exists(), index=False)
print(f"logged {len(folds_df)} fold rows -> {runs_path.name}")

## Summary: fold-level scores + bootstrap 95% CI

In [ ]:
rng = np.random.default_rng(0)

def boot_ci(v, n=CONFIG["N_BOOTSTRAP"]):
    v = np.asarray(v)
    bs = rng.choice(v, size=(n, len(v)), replace=True).mean(axis=1)
    return np.percentile(bs, [2.5, 97.5])

summary = []
for m in folds_df["model"].unique():
    for metric in ("f1_macro","mcc","bal_acc"):
        v = folds_df.loc[folds_df.model==m, metric].to_numpy()
        lo, hi = boot_ci(v)
        summary.append({"model": m, "metric": metric,
                        "mean": v.mean(), "std": v.std(ddof=1),
                        "ci_lo": lo, "ci_hi": hi})
summ = pd.DataFrame(summary).round(4)
print(summ.pivot(index="model", columns="metric",
                 values="mean").round(3).to_string())
summ.to_csv(OUT / f"T1_summary_{CONFIG['RUN_STAMP']}.csv", index=False)
summ[summ.metric == CONFIG["PRIMARY"]]

## Statistics: Friedman + Holm-corrected Wilcoxon, Cliff's delta, McNemar
Paired at fold level (same 75 folds across models). McNemar pooled over
out-of-fold predictions per (seed, repeat), exact binomial on discordant pairs.

In [ ]:
from scipy.stats import friedmanchisquare, wilcoxon, binomtest
from itertools import combinations

models_all = list(folds_df["model"].unique())
piv = folds_df.pivot_table(index=["seed","repeat","fold"], columns="model",
                           values=CONFIG["PRIMARY"])
fr = friedmanchisquare(*[piv[m] for m in models_all])
print(f"Friedman: chi2={fr.statistic:.2f} p={fr.pvalue:.2e}")

def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    gt = sum((x > b).sum() for x in a); lt = sum((x < b).sum() for x in a)
    return (gt - lt) / (len(a) * len(b))

pairs = list(combinations(models_all, 2))
raw = []
for m1, m2 in pairs:
    st = wilcoxon(piv[m1], piv[m2], zero_method="zsplit")
    raw.append({"pair": f"{m1} vs {m2}", "wilcoxon_p": st.pvalue,
                "cliffs_delta": cliffs_delta(piv[m1], piv[m2])})
raw.sort(key=lambda r: r["wilcoxon_p"])
mtests = len(raw)
for i, r in enumerate(raw):
    r["holm_p"] = min(1.0, r["wilcoxon_p"] * (mtests - i))
stats_df = pd.DataFrame(raw).round(5)
print(stats_df.to_string(index=False))

def mcnemar_pooled(m1, m2):
    b = c = 0
    for seed in CONFIG["SEEDS"]:
        for rep in range(CONFIG["N_REPEATS"]):
            p1, p2 = oof[(m1,seed,rep)], oof[(m2,seed,rep)]
            b += int(((p1 == y) & (p2 != y)).sum())
            c += int(((p1 != y) & (p2 == y)).sum())
    p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
    return b, c, p
b, c, p_mc = mcnemar_pooled("lgbm", "dummy_stratified")
print(f"\nMcNemar lgbm vs dummy_stratified: b={b} c={c} p={p_mc:.2e}")
stats_df.to_csv(OUT / f"T1_stats_{CONFIG['RUN_STAMP']}.csv", index=False)

## Permutation test (protocol v1.1)
Null: labels carry no descriptor-linked signal. 1000 permutations; the full
grouped-CV loop is rerun per permutation with FIXED modal HPs from the real
run (search inside every permutation is computationally prohibitive; fixing
HPs is conservative and disclosed). Applied to lgbm and logreg.

In [ ]:
def modal_params(plist):
    keys = plist[0].keys()
    return {k: Counter(p[k] for p in plist).most_common(1)[0][0] for k in keys}

perm_results = {}
rng_p = np.random.default_rng(0)
for mname in ("lgbm", "logreg"):
    est, _ = make_models()[mname]
    est.set_params(**modal_params(best_params_log[mname]))
    obs = folds_df.loc[folds_df.model == mname, CONFIG["PRIMARY"]].mean()

    null = []
    for i in range(CONFIG["N_PERMUTATIONS"]):
        yp_lab = rng_p.permutation(y)
        scores = []
        outer = StratifiedGroupKFold(n_splits=CONFIG["N_SPLITS"], shuffle=True,
                                     random_state=i)
        for tr, te in outer.split(X, yp_lab, groups):
            est.fit(X[tr], yp_lab[tr])
            scores.append(f1_score(yp_lab[te], est.predict(X[te]), average="macro"))
        null.append(np.mean(scores))
        if (i+1) % 100 == 0: print(f"  {mname}: {i+1}/{CONFIG['N_PERMUTATIONS']}")
    null = np.array(null)
    pval = (1 + (null >= obs).sum()) / (len(null) + 1)
    perm_results[mname] = {"observed": float(obs), "null_mean": float(null.mean()),
                           "null_p95": float(np.percentile(null,95)), "p_value": float(pval)}
    print(f"{mname}: observed={obs:.3f} | null mean={null.mean():.3f} "
          f"| null p95={np.percentile(null,95):.3f} | p={pval:.4f}")
(OUT / f"T1_permutation_{CONFIG['RUN_STAMP']}.json").write_text(_j.dumps(perm_results, indent=2))

## Frozen failure-criterion evaluation (§P7 — automatic verdict)
Success requires ALL of: (1) Holm-corrected p < 0.05 for best model vs
stratified dummy, (2) Cliff's delta >= 0.33, (3) permutation p < 0.05.
The verdict prints itself; there is no interpretive wiggle room.

In [ ]:
best_model = summ[(summ.metric==CONFIG["PRIMARY"])].sort_values("mean").iloc[-1]["model"]
row = stats_df[stats_df["pair"].str.contains(best_model)
               & stats_df["pair"].str.contains("dummy_stratified")].iloc[0]
crit = {
    "best_model": best_model,
    "holm_p_vs_dummy": float(row["holm_p"]),
    "cliffs_delta_vs_dummy": float(abs(row["cliffs_delta"])),
    "permutation_p": perm_results.get(best_model, perm_results["lgbm"])["p_value"],
}
crit["PASS"] = (crit["holm_p_vs_dummy"] < 0.05
                and crit["cliffs_delta_vs_dummy"] >= 0.33
                and crit["permutation_p"] < 0.05)
print(_j.dumps(crit, indent=2))
if crit["PASS"]:
    print("\nVERDICT: T1 SUCCESS CRITERION MET — proceed to interpretability (C2).")
else:
    print("\nVERDICT: T1 FAILURE CRITERION TRIGGERED — per protocol, T1 is a "
          "negative finding; the paper reframes accordingly. No metric shopping.")
(OUT / f"T1_verdict_{CONFIG['RUN_STAMP']}.json").write_text(_j.dumps(crit, indent=2))

## Environment snapshot

In [ ]:
import subprocess, sys
freeze = subprocess.run([sys.executable,"-m","pip","freeze"],
                        capture_output=True, text=True).stdout
(OUT / f"environment_T1_{CONFIG['RUN_STAMP']}.txt").write_text(freeze)
print("environment snapshot saved |", len(freeze.splitlines()), "packages")

## Next
- Paste the summary table, stats table, permutation lines and the verdict JSON
  back for review (F5 / Step 2: interpretation + go/no-go on C2 analysis).
- Graph models (CGCNN/ALIGNN, GPU) follow in F6 only if T1 signal exists.